# ♻️ Recycling Warehouse: Automated Tile Inspection

Reclaimed floor tiles arrive at a recycling warehouse by the thousand. Instead of a person checking every tile, each one rides a conveyor belt through an **inspection station**: a camera photographs it and a computer-vision model decides whether it can be sold. In this project you build that model.

▶ **Start here:** run the next two cells. The first fetches the project files, the second shows the inspection line you are about to build.

In [ ]:
# @title 🔁 Clone the project (run after every restart) { display-mode: "form" }
# 🔁 Run after every restart — clone the repository (skipped if it is already there)
import os
import subprocess
import sys

REPO_URL = "https://github.com/eth-bmai-hs26/we-cv-new-projects-public.git"
BRANCH = "main"
REPO_DIR = "we-cv-new-projects-public"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start="."):
    path = os.path.abspath(start)
    while path != os.path.dirname(path):
        if os.path.isdir(os.path.join(path, "week3", "project1")):
            return path
        path = os.path.dirname(path)
    return None


def clone(repo_url, branch, target):
    result = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, repo_url, target], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Could not clone {repo_url} (branch {branch}):\n{result.stderr}")


try:
    os.getcwd()
except FileNotFoundError:                           # the folder we were in was deleted (e.g. to re-clone)
    os.chdir("/content" if IN_COLAB else os.path.expanduser("~"))

root = find_repo_root()
if root is None:                                    # not inside the repo yet → clone it
    if not os.path.isdir(REPO_DIR):
        clone(REPO_URL, BRANCH, REPO_DIR)
    root = os.path.abspath(REPO_DIR)

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)
print("Working directory:", os.getcwd())

In [ ]:
# @title ▶ Show the inspection line you will build { display-mode: "form" }
from week3.project1.widgets import assembly_line_intro
assembly_line_intro()

**The problem.** Every tile photo must answer two questions:
1. *Which part of the tile is still usable?* → **Segmentation** (U-Net): every pixel is *belt*, *usable tile surface* or *damaged zone* (cracks and chips plus a small cutting margin).
2. *Can the tile be sold?* → **Decision**: if at least **85 % of the tile's surface is usable → APPROVE** (sell, cut down if needed), otherwise **REJECT** (recycle).

| | Section | What you do |
|---|---|---|
| 1 | 🗂️ **Setup** | pick a GPU, choose where your work is saved |
| 2 | 🔍 **Dataset** | write a `Dataset`, cache 5 000 tile photos with their class maps, look at them |
| 3 | 🧱 **U-Net** | build and train a first 3-class segmentation model |
| 4 | 🔢 **FNN head** | classify from a *pooled* latent vector |
| 5 | 🧠 **CNN head** | classify from the *spatial* latent map |
| 6 | ⚖️ **Evaluation** | scoreboard, confusion matrix, error inspector |
| 7 | 🚀 **Inference + game** | one tile through the pipeline, then play human vs. model |
| 8 | 🔭 **Receptive field** | *why* a small network confuses cracks with stains — an experiment |
| 9 | 🏆 **Challenge** | improve every model and beat the targets (incl. a new **usable-share regressor**) |
| 10 | 🎁 **Bonus: transfer learning** | a U-Net with an ImageNet-pretrained ResNet encoder |
| 11 | 🏭 **Deploy** | download your best model and run it in the warehouse inspection platform |

Every section answers one question and hands its answer to the next — look for the 🧭 **Why this step?** notes. Most results appear as **interactive panels** (live training dashboards, sliders, clickable matrices). They are plain HTML, so they also stay visible when you save and share the notebook.

---
## 1 🗂️ Setup

**Runtime.** In Colab choose *Runtime → Change runtime type → GPU* (with Colab Pro pick **L4** or **A100**, and turn on **High-RAM**). The whole notebook needs roughly 45–60 minutes of GPU time.

**Coming back days later?** Every model you train is saved — to your Google Drive in Colab, so it survives runtime restarts. After a restart:
1. run the cells marked **🔁** in Sections 1–2 (about two minutes: clone, imports, storage, data),
2. jump to the section you want to continue and run its **▶ RESUME HERE** cell — it reloads the saved models instead of retraining them.

Import the libraries and the project helpers (`week3/project1/*.py`).

In [ ]:
# @title 🔁 Load the libraries and project helpers (run after every restart) { display-mode: "form" }
# 🔁 Run after every restart — libraries and project helpers
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader

try:
    import dill  # noqa: F401 — used to save whole models (weights + class code)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dill"], check=True)

from week3.project1.data import (build_cache, make_splits, subsample, CachedTileDataset,
                                 USABLE_THRESHOLD, usable_fraction, CLASS_NAMES)
from week3.project1.utils import (
    LatentUNetBase, DepthUNet, dice_loss, get_device, predict_classes, usable_from_logits, count_parameters,
    train_segmentation, train_head, evaluate_models, predictions_table, plot_segmentation_results,
    theoretical_receptive_field, effective_receptive_field, defect_sizes,
    set_checkpoint_dir, save_model, load_model, load_models, class_from_checkpoint, has_checkpoint,
    export_for_platform,
)
from week3.project1 import widgets as W
from week3.project1.app_colab import launch_warehouse_app

# Instructors: FAST_DEV_RUN=1 runs the whole notebook on a few hundred images for 1 epoch.
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "0") == "1"
def EPOCHS(n):
    return 1 if FAST_DEV_RUN else n

CACHE_DIR = "/content/cache" if IN_COLAB else "cache"
device = get_device()
torch.manual_seed(0)
print(f"Device: {device}" + ("   (FAST_DEV_RUN)" if FAST_DEV_RUN else ""))
if device.type == "cpu":
    print("⚠️ No GPU found. In Colab: Runtime → Change runtime type → GPU.")

**Where your work is saved.** In Colab the next cell mounts your Google Drive (Colab asks for permission once) and keeps checkpoints and the scoreboard in `MyDrive/warehouse-inspector/`. Anything stored only in `/content` disappears when the runtime is recycled.

In [ ]:
# @title 🔁 Choose where your work is saved (run after every restart) { display-mode: "form" }
# 🔁 Run after every restart — where checkpoints and the scoreboard live
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/warehouse-inspector"
else:
    WORK_DIR = CACHE_DIR
set_checkpoint_dir(f"{WORK_DIR}/checkpoints")

board = W.Scoreboard(targets={"Threshold": 96, "FNN": 78, "CNN": 92, "Regression": 90},
                     iou_target=0.90, mae_target=2.5, store=f"{WORK_DIR}/attempts.json")

saved = sorted(f[:-3] for f in os.listdir(f"{WORK_DIR}/checkpoints") if f.endswith(".pt"))
print(f"Saving to {WORK_DIR}")
print("Saved models: " + (", ".join(saved) if saved else "none yet"))

### 1.1 Dice loss

Besides **cross-entropy** you will meet the **Dice loss** (imported from `utils.py`). For each class it measures the overlap between the predicted region $P$ and the true region $T$ as a whole, instead of judging every pixel independently, and averages over the three classes:

$$\text{Dice} = \frac{2\,|P \cap T|}{|P| + |T|} \qquad \text{loss} = 1 - \text{Dice}$$

It is useful when classes are unbalanced — the damaged zone is often only a few percent of the image — and it rewards getting whole regions right.

### 1.2 📏 The metrics you will meet

Every result in this notebook is reported with the same handful of numbers. If one of them is new to you, this is the place to look it up.

**How good is the decision?** Computed per tile, on the held-out **test set** (750 tiles the models never saw during training; the *validation* set is used during training to pick settings).

| Metric | What it measures | How to read it |
|---|---|---|
| **Accuracy** | share of test tiles where APPROVE / REJECT matches the truth | 97 % = 3 wrong decisions per 100 tiles |
| **Confusion matrix** | the four kinds of outcome: correct approvals, correct rejections, **false approvals** (a damaged tile is sold) and **false rejections** (a sellable tile is recycled) | accuracy treats both errors alike; the business does not (Section 9.6) |
| **Business cost** | cost of a false approval × their number + cost of a false rejection × their number | you choose the two costs; the threshold with the lowest total is the best operating point |
| **Probability *p*** | what the FNN / CNN heads output: how sure they are that a tile should be approved | APPROVE when *p* > 0.5 |

**How good is the segmentation?** Computed per pixel, for each of the three classes (belt, usable, damaged).

| Metric | What it measures | How to read it |
|---|---|---|
| **IoU** (intersection over union, Jaccard index) | the overlap of the predicted region and the true region, divided by their combined area (formula below) | 1 = perfect, 0 = no overlap; unlike pixel accuracy it is not fooled by a large, easy background |
| **Mean IoU** | the IoU of each class, averaged over the three classes | the headline segmentation number; all test pixels are pooled before dividing |
| **Damaged-zone IoU** | the IoU of the *damaged* class alone | the hardest class: small, thin and easy to confuse with stains |
| **Dice** | twice the overlap divided by the sum of both areas; closely related to IoU (Dice = 2·IoU / (1 + IoU)) | used here as a **loss** (1 − Dice), not as a reported score |

With $P$ the predicted and $T$ the true region of one class:

$$\text{IoU} = \frac{|P \cap T|}{|P \cup T|} \qquad\qquad \text{Dice} = \frac{2\,|P \cap T|}{|P| + |T|}$$

**How good is the number the decision is based on?**

| Metric | Definition | How to read it |
|---|---|---|
| **Usable share** | usable pixels ÷ (usable + damaged pixels) — the share of the tile that can be sold | the business rule: APPROVE when it is at least **85 %** |
| **Usable-share error (MAE)** | mean absolute error: the average of \|predicted − true usable share\| over all test tiles, in **percentage points (pp)** | 2 pp means the estimate is off by 2 points on average — e.g. 84 % instead of 86 %, which can flip a decision near the line |

**Losses** (what training minimises — lower is better, but the values are not comparable between different losses):
- **Cross-entropy** — the standard loss for classifying each pixel into one of several classes (U-Net).
- **Binary cross-entropy (BCE)** — the same idea for a single yes/no output (FNN and CNN heads).
- **L1 loss** — the mean absolute difference between a predicted and a true number (usable-share regressor).

**Receptive field** (Section 8) — the size, in input pixels, of the region that can influence one unit of the network. *Theoretical*: every pixel that is connected to the unit at all. *Effective*: the part that actually carries most of the influence in the trained network (usually much smaller).

---
## 2 🔍 Dataset

We have **5 000 photos from the inspection camera**, one tile per photo, each with two masks:

| File | What it shows |
|---|---|
| `dataset_new/original/tile_0001.jpg` | the photo: one tile on the conveyor belt |
| `dataset_new/tiles/tile_0001-tiles.png` | white = every pixel of the tile (damaged or not) |
| `dataset_new/segmented/tile_0001-segmented.png` | white = the **usable** part of the tile |
| `dataset_new/ground_truth.csv` | per tile: usable share, damage type, material … |

Together they give every pixel one of **three classes**:

| Class | Meaning | How to get it |
|---|---|---|
| 0 · background | the conveyor belt | outside the tile mask |
| 1 · usable | tile surface that can be sold | inside the usable mask |
| 2 · damaged | what a cutter would remove: cracks and chips **plus a small safety margin** | inside the tile mask but not in the usable mask |

A tile is approved when its **usable share** = usable ÷ (usable + damaged) is at least **85 %**.

Look closely at the photos later: many tiles carry **stains, dirt, scratches or marble veins** that are *not* damage. Telling them apart from real cracks becomes the key question of Section 8.

### 2.1 A `Dataset` class

A PyTorch `Dataset` answers two questions: *how many samples are there?* (`__len__`) and *give me sample i* (`__getitem__`). Photos are loaded as **grayscale** and resized to **256×256**.

In [ ]:
class SegmentationDataset(Dataset):
    """
    Tile photos and their 3-class maps, read from disk.

        root/
            original/    ← photos           tile_0001.jpg
            tiles/       ← tile masks       tile_0001-tiles.png       (white = tile)
            segmented/   ← usable masks     tile_0001-segmented.png   (white = usable)

    Returns (image, target):
        image:  float tensor (1, H, W) in [0, 1]
        target: long tensor (H, W) with 0 = background, 1 = usable, 2 = damaged
    """

    def __init__(self, root, image_size=(256, 256)):
        super().__init__()
        self.root = root
        self.images = sorted(f for f in os.listdir(os.path.join(root, "original")) if f.endswith(".jpg"))

        # TODO: Define two torchvision.transforms.Compose pipelines:
        #   self.img_transform  : resize to image_size, then convert to a tensor
        #   self.mask_transform : the same, but resize with NEAREST interpolation
        # Hint: T.Resize(size, interpolation=T.InterpolationMode.NEAREST) and T.ToTensor()
        # Why NEAREST for masks? Bilinear resizing would invent grey values at the edges.
        self.img_transform = ...   # YOUR CODE HERE
        self.mask_transform = ...  # YOUR CODE HERE

    def _path(self, folder, img_name):
        """'tile_0001.jpg' → root/<folder>/tile_0001-<folder>.png"""
        stem, _ = os.path.splitext(img_name)
        return os.path.join(self.root, folder, f"{stem}-{folder}.png")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        # TODO: Load the photo and both masks as grayscale images
        # Hint: Image.open(os.path.join(self.root, "original", img_name)).convert("L")
        #       Image.open(self._path("tiles", img_name)).convert("L")
        img = ...     # YOUR CODE HERE
        tile = ...    # YOUR CODE HERE
        usable = ...  # YOUR CODE HERE

        # TODO: Apply the matching transform to each, then turn both masks into boolean tensors (> 0.5)
        img = ...     # YOUR CODE HERE
        tile = ...    # YOUR CODE HERE  → shape (H, W), dtype bool   (hint: [0] drops the channel dim)
        usable = ...  # YOUR CODE HERE

        # TODO: Build the class map: start with zeros (background), set tile pixels to 2 (damaged),
        #       then set usable pixels to 1. Hint: torch.zeros(tile.shape, dtype=torch.long)
        target = ...  # YOUR CODE HERE
        return img, target

In [ ]:
DATA_ROOT = "week3/project1/dataset_new"
ds_disk = SegmentationDataset(DATA_ROOT, image_size=(256, 256))
img, target = ds_disk[0]
print(f"{len(ds_disk)} samples · image {tuple(img.shape)} · target {tuple(target.shape)} · classes {target.unique().tolist()}")

### 2.2 Caching the data in memory

Your `Dataset` decodes three image files **every time** a sample is requested — about 10 000 decodes per epoch, repeated for every epoch. With Colab's RAM we can decode everything **once** and keep compact tensors in memory (≈ 650 MB). `build_cache` does exactly what your class does, for all images at once; `CachedTileDataset` then serves `(image, target, label)` straight from memory.

The first run decodes all images (about a minute); later runs in the same runtime load the cached file.

In [ ]:
# @title 🔁 Load all 5 000 tiles into memory (run after every restart) { display-mode: "form" }
# 🔁 Run after every restart — all 5 000 photos and class maps in memory
DATA_ROOT = "week3/project1/dataset_new"
cache = build_cache(DATA_ROOT, size=256, cache_path=f"{CACHE_DIR}/tiles_256.pt")

How much faster is it, and does your `SegmentationDataset` produce exactly the same tensors as the cache?

In [ ]:
# @title Compare speed, and check your Dataset against the cache { display-mode: "form" }
t0 = time.time()
_ = [ds_disk[i] for i in range(100)]
disk_ms = 10 * (time.time() - t0)

t0 = time.time()
_ = [CachedTileDataset(cache)[i] for i in range(100)]
mem_ms = 10 * (time.time() - t0)
print(f"From disk: {disk_ms:.1f} ms/sample · from memory: {mem_ms:.2f} ms/sample "
      f"→ {disk_ms / max(mem_ms, 1e-3):.0f}× faster, ~{3500 * (disk_ms - mem_ms) / 1000:.0f} s saved per epoch")

W.check_dataset(ds_disk, cache);

### 2.3 Labels and the train / validation / test split

The label of a tile follows from its class map: **APPROVE if at least 85 % of the tile is usable**. We compute it from the 256×256 maps the model actually sees (`ground_truth.csv` was computed at full resolution; resizing moves a few tiles across the line, see the printout).

We split once, reproducibly, into **70 % train / 15 % validation / 15 % test**. The test set is only used for the final evaluation.

In [ ]:
# @title 🔁 Split into ds_train · ds_train_aug · ds_val · ds_test (run after every restart) { display-mode: "form" }
# 🔁 Run after every restart — splits (always the same) and datasets
train_idx, val_idx, test_idx = make_splits(len(cache), fracs=(0.70, 0.15, 0.15), seed=42)
if FAST_DEV_RUN:
    train_idx, val_idx, test_idx = subsample(train_idx, 256), subsample(val_idx, 96), subsample(test_idx, 160)

ds_train = CachedTileDataset(cache, train_idx)
ds_train_aug = CachedTileDataset(cache, train_idx, augment=True)   # random flips / 90° rotations (Section 9)
ds_val = CachedTileDataset(cache, val_idx)
ds_test = CachedTileDataset(cache, test_idx)

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True)

# a damaged test tile close to the 85 % line, used as the running example in Sections 7 and 8
border = [i for i in ds_test.indices if abs(cache.usable[i] - USABLE_THRESHOLD) < 0.05 and cache.usable[i] < 1]
IDX = border[len(border) // 2] if border else ds_test.indices[0]

print(f"Train {len(ds_train):,} · validation {len(ds_val):,} · test {len(ds_test):,}")
print(f"{int(cache.labels.sum()):,} APPROVE / {int(len(cache) - cache.labels.sum()):,} REJECT overall")
print(f"{100 * np.mean(cache.usable >= 0.999):.0f} % of tiles are pristine (no damage at all)")
print(f"{cache.label_flips} labels ({100 * cache.label_flips / len(cache):.1f} %) differ between the full-resolution CSV and the 256 px maps")
print(f"{100 * np.mean(np.abs(cache.usable - USABLE_THRESHOLD) < 0.05):.0f} % of tiles lie within ±5 points of the 85 % line — these are the hard ones")

In [ ]:
# @title Show sample tiles { display-mode: "form" }
W.dataset_preview(cache, ds_test.indices, n=8)

#### What to look for
- Grey = belt, white = usable surface, red = damaged zone. The red zone is a bit wider than the crack itself — that is the cutting margin.
- Find a tile with a **stain or marble vein** and check its class map: it stays white. Only cracks and chips count.
- Tiles close to 85 % are hard even for a human — try it in the game in Section 7.

---
## 3 🧱 A first segmentation model

> 🧭 **Why this step?** Before we can judge a tile we need to know *where* its usable surface is. We start with the simplest possible encoder–decoder to get a **baseline** — every later step is measured against it.

We start with a simple **encoder–decoder**. The encoder compresses the photo into a small **bottleneck** representation; the decoder expands it back into a class map. The last layer produces **three scores (logits) per pixel**, one per class; the predicted class is the one with the highest score.

**Architecture (`base_ch = 8`, input 256×256)**

| Stage | Module | Output shape |
|---|---|---|
| Input | — | (B, 1, 256, 256) |
| `enc1` | Conv 3×3 (1 → 8) + ReLU | (B, 8, 256, 256) |
| `down1` | Conv 4×4, stride 2 | (B, 8, 128, 128) |
| `enc2` | Conv 3×3 (8 → 16) + ReLU | (B, 16, 128, 128) |
| `down2` | Conv 4×4, stride 2 | (B, 16, 64, 64) |
| **`bottleneck`** | Conv 3×3 (16 → 16) + ReLU | **(B, 16, 64, 64)** |
| `up1` | ConvTranspose 4×4, stride 2 (16 → 8) | (B, 8, 128, 128) |
| `dec1` | Conv 3×3 (8 → 8) + ReLU | (B, 8, 128, 128) |
| `up2` | ConvTranspose 4×4, stride 2 | (B, 8, 256, 256) |
| `dec2` | Conv 3×3 (8 → 8) + ReLU | (B, 8, 256, 256) |
| `out_conv` | Conv 1×1 (8 → 3) | (B, 3, 256, 256) — one **logit per class** per pixel |

Every model in this notebook inherits from `LatentUNetBase` and implements two methods:
- `encode(x)` → `(bottleneck, skips)` — the bottleneck map plus a list of encoder features for skip connections (empty here),
- `decode(bottleneck, skips)` → class logits `(B, 3, H, W)`.

The base class turns these into `forward(x)`, `get_latent_map(x)` (the bottleneck map, **(B, 16, 64, 64)**) and `get_latent(x)` (the map averaged over space, **(B, 16)**). The classifier heads in Sections 4 and 5 use the last two.

In [ ]:
class SimpleConvBlock(nn.Module):
    """Convolution block: Conv 3×3 → ReLU"""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        # TODO: self.block = nn.Sequential(...) with
        #   1. nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        #   2. nn.ReLU(inplace=True)
        self.block = ...  # YOUR CODE HERE

    def forward(self, x):
        return self.block(x)


class SimpleUNet(LatentUNetBase):
    """Simple encoder–decoder (no skip connections) — see the table above."""

    def __init__(self, in_ch=1, base_ch=8, n_classes=3):
        super().__init__()
        self.latent_dim = base_ch * 2          # channels in the bottleneck

        # TODO: Encoder — follow the table
        #   self.enc1  = SimpleConvBlock(in_ch, base_ch)
        #   self.down1 = nn.Conv2d(base_ch, base_ch, kernel_size=4, stride=2, padding=1)   # halves H and W
        #   self.enc2  = SimpleConvBlock(base_ch, base_ch * 2)
        #   self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, kernel_size=4, stride=2, padding=1)
        # YOUR CODE HERE

        # TODO: Bottleneck — SimpleConvBlock(base_ch * 2, base_ch * 2)
        self.bottleneck = ...  # YOUR CODE HERE

        # TODO: Decoder — mirror of the encoder
        #   self.up1  = nn.ConvTranspose2d(base_ch * 2, base_ch, kernel_size=4, stride=2, padding=1)  # doubles H and W
        #   self.dec1 = SimpleConvBlock(base_ch, base_ch)
        #   self.up2  = nn.ConvTranspose2d(base_ch, base_ch, kernel_size=4, stride=2, padding=1)
        #   self.dec2 = SimpleConvBlock(base_ch, base_ch)
        #   self.out_conv = nn.Conv2d(base_ch, n_classes, kernel_size=1)   # one score per class
        # YOUR CODE HERE

    def encode(self, x):
        # TODO: enc1 → down1 → enc2 → down2 → bottleneck
        # Return (bottleneck_output, []) — the empty list means "no skip connections".
        raise NotImplementedError  # YOUR CODE HERE

    def decode(self, b, skips):
        # TODO: up1 → dec1 → up2 → dec2 → out_conv (skips is empty and unused)
        raise NotImplementedError  # YOUR CODE HERE

In [ ]:
torch.manual_seed(0)
unet = SimpleUNet(in_ch=1, base_ch=8).to(device)
W.architecture_diagram(unet, title="SimpleUNet")

### 3.1 Training the U-Net

> **📝 The PyTorch training loop.** Every training step follows the same five steps:
> 1. **Forward pass** — `pred = model(img)`
> 2. **Loss** — how wrong is the prediction?
> 3. **Zero gradients** — `optimizer.zero_grad()` (otherwise they accumulate)
> 4. **Backward pass** — `loss.backward()` computes the gradients
> 5. **Update** — `optimizer.step()` changes the weights
>
> You will write this loop three times (U-Net, FNN, CNN).

We train with **cross-entropy** — the standard multi-class loss: `nn.CrossEntropyLoss` takes the raw logits `(B, 3, H, W)` and the class map `(B, H, W)` and applies the softmax internally — and the **Adam** optimiser. The label returned by the dataset is ignored here; segmentation only needs `(image, target)`. The dashboard redraws after every epoch.

In [ ]:
def train_unet(model, dataloader, epochs=10, lr=1e-3):
    """Train a 3-class segmentation model with cross-entropy and Adam."""
    model.to(device)

    # TODO: Define the loss and the optimizer
    #   - Loss: nn.CrossEntropyLoss()   (logits (B, 3, H, W) vs class map (B, H, W))
    #   - Optimizer: torch.optim.Adam(model.parameters(), lr=lr)
    criterion = ...  # YOUR CODE HERE
    optimizer = ...  # YOUR CODE HERE

    dash = W.TrainingDashboard("Simple U-Net", epochs)
    for ep in range(epochs):
        model.train()
        epoch_loss = 0.0
        for img, target, _label in dataloader:
            img, target = img.to(device), target.to(device)

            # TODO: the five steps — forward pass, loss, zero_grad, backward, step
            # YOUR CODE HERE

            epoch_loss += loss.item()
        dash.update(ep + 1, epochs, train_loss=epoch_loss / len(dataloader))


train_unet(unet, dl_train, epochs=EPOCHS(10), lr=1e-3)

In [ ]:
# @title 💾 Save the model { display-mode: "form" }
save_model(unet, "simple_unet")     # reloaded by the ▶ RESUME HERE cells of later sections

#### What to look for
- Does the loss still go down at the end, or has it flattened?
- In the predictions below: are the edges of the tile sharp or blurry? Are **cracks** found, and are **stains or veins** wrongly marked as damaged?

In [ ]:
# @title Show predictions on test tiles { display-mode: "form" }
plot_segmentation_results(unet, ds_test, n=8, title="SimpleUNet on test tiles: photo / truth / prediction")

---
## 4 🔢 FNN classifier on the pooled latent vector

> 🧭 **Why this step?** The segmentation gives a map; the business needs a yes/no. Can a tiny classifier read the decision straight from what the U-Net has learned, without counting pixels? We try the cheapest version first: average the bottleneck into one vector.

The first classifier collapses the bottleneck into one vector with **Global Average Pooling** and feeds it to a feed-forward network:

```
bottleneck map (B, 16, 64, 64) ── average over H, W ──► latent vector (B, 16) ── Linear(16 → 1) + Sigmoid ──► P(APPROVE)
```

> **🔑 Freezing the backbone.** In Sections 4 and 5 the U-Net is **frozen**: we only train the small head on top. This is **transfer learning** — the U-Net learned useful features for segmentation and we reuse them for a different task. In code: `unet.eval()` and computing the features inside `torch.no_grad()`, so no gradients flow into the U-Net.
>
> Why freeze? It prevents overfitting a small head's task onto the big backbone, it is much cheaper (no backward pass through the U-Net), and it preserves what the backbone already knows.

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet" not in globals():
    unet = load_model("simple_unet")

In [ ]:
class SimpleApprovalFNN(nn.Module):
    """Linear(latent_dim → 1) + Sigmoid on the pooled latent vector."""

    def __init__(self, latent_dim=16):
        super().__init__()
        # TODO: self.net = nn.Sequential(nn.Linear(latent_dim, 1), nn.Sigmoid())
        self.net = ...  # YOUR CODE HERE

    def forward(self, z):
        # TODO: return self.net(z) with the last dimension squeezed → shape (B,)
        return ...  # YOUR CODE HERE


def train_classifier_fnn(unet, classifier, dataloader, epochs=10, lr=1e-2):
    """Train the FNN head; the U-Net stays frozen."""
    unet.eval()
    classifier.to(device)

    # TODO: loss = nn.BCELoss() (inputs are already probabilities)
    #       optimizer = torch.optim.SGD on the CLASSIFIER parameters only
    criterion = ...  # YOUR CODE HERE
    optimizer = ...  # YOUR CODE HERE

    dash = W.TrainingDashboard("FNN head (pooled latent)", epochs)
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _targets, labels in dataloader:
            imgs, labels = imgs.to(device), labels.float().to(device)

            # TODO: latent vector from the frozen U-Net:  with torch.no_grad(): z = unet.get_latent(imgs)
            # YOUR CODE HERE

            # TODO: forward pass through the classifier, loss, zero_grad, backward, step
            # YOUR CODE HERE

            total_loss += loss.item()
        dash.update(ep + 1, epochs, train_loss=total_loss / len(dataloader))

In [ ]:
torch.manual_seed(0)
classifier_fnn = SimpleApprovalFNN(latent_dim=unet.latent_dim)
train_classifier_fnn(unet, classifier_fnn, dl_train, epochs=EPOCHS(10), lr=1e-2)
save_model(classifier_fnn, "simple_fnn")

---
## 5 🧠 CNN classifier on the spatial latent map

> 🧭 **Why this step?** Averaging throws away *where* things are. If a head that keeps the spatial map beats the FNN, spatial layout matters for the decision — which tells us how to design heads later.

The second classifier **skips the early pooling**: it receives the full bottleneck map and processes it with a convolution *before* aggregating.

| Layer | Input | Output | Purpose |
|---|---|---|---|
| `Conv2d(16 → 4, 3×3, pad 1)` + ReLU | (B, 16, 64, 64) | (B, 4, 64, 64) | spatial processing |
| `AdaptiveAvgPool2d(1)` | (B, 4, 64, 64) | (B, 4, 1, 1) | global aggregate |
| `Flatten` → `Linear(4 → 1)` → `Sigmoid` | (B, 4) | (B,) | probability |

> **🤔 FNN vs CNN.** Pooling first throws away *where* things are — the FNN only knows the average feature. The CNN can look at spatial patterns first ("this line runs across the tile — a crack, not a stain") and only then summarise.

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet" not in globals():
    unet = load_model("simple_unet")

In [ ]:
class SimpleApprovalCNN(nn.Module):
    """Conv 3×3 (in → 4) + ReLU → global average pool → Linear(4 → 1) + Sigmoid."""

    def __init__(self, in_channels=16):
        super().__init__()
        # TODO: self.features = nn.Sequential(nn.Conv2d(in_channels, 4, kernel_size=3, padding=1), nn.ReLU(inplace=True))
        self.features = ...  # YOUR CODE HERE
        # TODO: self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(4, 1), nn.Sigmoid())
        self.classifier = ...  # YOUR CODE HERE

    def forward(self, z_map):
        # TODO: features → classifier, squeeze the last dimension → (B,)
        return ...  # YOUR CODE HERE


def train_classifier_cnn(unet, classifier, dataloader, epochs=10, lr=1e-2):
    """Train the CNN head; the U-Net stays frozen."""
    unet.eval()
    classifier.to(device)

    # TODO: nn.BCELoss() and torch.optim.SGD(classifier.parameters(), lr=lr)
    criterion = ...  # YOUR CODE HERE
    optimizer = ...  # YOUR CODE HERE

    dash = W.TrainingDashboard("CNN head (spatial latent map)", epochs)
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _targets, labels in dataloader:
            imgs, labels = imgs.to(device), labels.float().to(device)

            # TODO: the spatial latent MAP this time: unet.get_latent_map(imgs), inside torch.no_grad()
            # YOUR CODE HERE

            # TODO: the same five-step pattern as the FNN
            # YOUR CODE HERE

            total_loss += loss.item()
        dash.update(ep + 1, epochs, train_loss=total_loss / len(dataloader))

In [ ]:
torch.manual_seed(0)
classifier_cnn = SimpleApprovalCNN(in_channels=unet.latent_dim)
train_classifier_cnn(unet, classifier_cnn, dl_train, epochs=EPOCHS(10), lr=1e-2)
save_model(classifier_cnn, "simple_cnn")

---
## 6 ⚖️ Evaluation on the test set

> 🧭 **Why this step?** Now we measure all three methods on tiles the models have **never seen**, and look at *which* tiles fail. The failure cases tell us what to fix next.

We compare three ways to decide on the **held-out test set**:

| Method | What it sees | How it decides |
|---|---|---|
| **Threshold** | the predicted class map | usable ÷ (usable + damaged) pixels, apply the 85 % rule — no extra training |
| **FNN** | pooled latent vector | learned from labels |
| **CNN** | spatial latent map | learned from labels |

The **Scoreboard** keeps every attempt you record, so you can track your progress through the challenge.

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3), the FNN head (§4), the CNN head (§5) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "classifier_cnn" not in globals():
    unet, classifier_fnn, classifier_cnn = load_models("simple_unet", "simple_fnn", "simple_cnn")

In [ ]:
res_basic = evaluate_models(unet, classifier_fnn, classifier_cnn, ds_test, label="Basic models")
board.record(res_basic)

In [ ]:
# @title Show the first predictions { display-mode: "form" }
predictions_table(res_basic, n=10)

**Where do the errors come from?** Click the cells of the confusion matrix, then step through the inspector. The error map colours pixels the U-Net got wrong: **red** = damage it missed (that surface would be sold), **blue** = good surface it marked as damaged (wasted), **amber** = tile and belt confused.

> ⚠️ **A pitfall of the metric.** The usable share only counts pixels the model assigns to the *tile*. If it mistakes part of a tile for belt (amber), those pixels drop out of both the numerator and the denominator — so a tile the model barely found can still look "100 % usable". The mean IoU and the amber areas in the error maps reveal this; watch for it in the basic model.

In [ ]:
# @title Show where the models go wrong { display-mode: "form" }
W.confusion_explorer(res_basic)

In [ ]:
# @title Open the test-set inspector { display-mode: "form" }
W.test_inspector(res_basic, max_items=60)

#### Questions
1. Which method is best, and is any of them good enough for a warehouse?
2. Look at the red and blue regions in the error maps. Which **cracks** does the U-Net miss, and which **stains, veins or scratches** does it mistake for damage?
3. The threshold rule needs no training at all — so why can it still be bad?

---
## 7 🚀 One tile through the full pipeline

> 🧭 **Why this step?** Accuracy numbers hide what a single decision looks like. Following one borderline tile through the pipeline — and then playing against the model — builds intuition for how hard the task is.

A damaged tile close to the 85 % line — the case where the decision matters most. `forward_all` returns the class logits **and** the bottleneck in one encoder pass.

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3), the FNN head (§4), the CNN head (§5) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "classifier_cnn" not in globals():
    unet, classifier_fnn, classifier_cnn = load_models("simple_unet", "simple_fnn", "simple_cnn")
if "res_basic" not in globals():
    res_basic = evaluate_models(unet, classifier_fnn, classifier_cnn, ds_test, label="Basic models")

In [ ]:
img = (cache.images[IDX].float() / 255.0).unsqueeze(0).to(device)     # (1, 1, 256, 256)
unet.eval(); classifier_fnn.eval(); classifier_cnn.eval()
with torch.no_grad():
    logits, z_map = unet.forward_all(img)       # class logits (1, 3, H, W) + bottleneck map
    pred_map = logits.argmax(dim=1)[0]          # predicted class per pixel: 0 belt, 1 usable, 2 damaged
    usable = usable_fraction(pred_map).item()   # usable ÷ (usable + damaged)
    fnn_prob = classifier_fnn(z_map.mean(dim=(2, 3))).item()   # same as unet.get_latent(img)
    cnn_prob = classifier_cnn(z_map).item()

decide = lambda ok: "APPROVE" if ok else "REJECT"
W.pipeline_card(
    cache.image(IDX), cache.target(IDX), pred_map.cpu(),
    {"Threshold": (decide(usable >= USABLE_THRESHOLD), f"{100 * usable:.1f}% usable"),
     "FNN": (decide(fnn_prob > 0.5), f"p = {fnn_prob:.2f}"),
     "CNN": (decide(cnn_prob > 0.5), f"p = {cnn_prob:.2f}")},
    true_usable=float(cache.usable[IDX]), pred_usable=usable,
    title=f"{cache.files[IDX]} — basic models")

#### How to read this card

This tile was picked because it sits **close to the 85 % line**, where decisions are hardest. Expect the methods to disagree here, and ask *why*:

- **The threshold verdict follows the U-Net's estimate.** Compare "Truth · …% usable" with "U-Net · …% usable". Near the line, an error of just one or two points is enough to flip APPROVE into REJECT (or the other way round).
- **Check the error map before trusting the estimate.** Amber pixels are tile the model took for belt (or belt it took for tile, e.g. the side rail). They drop out of, or sneak into, the usable share, so an estimate can land close to the truth *by luck*. Red and blue show missed damage and wasted good surface.
- **Look at the heads' probabilities, not only their verdicts.** A *p* close to 0.5 means the FNN or CNN is unsure; a right answer with *p* = 0.55 is closer to a coin flip than to a confident decision.

This is why Section 9 pushes the usable-share error down to a point or two, and why the inspection platform (Section 11) sends tiles within ±5 points of the line to a **human review queue** instead of deciding alone.

### 7b 🌐 Warehouse Inspector — you vs. the model

**Challenge mode:** look at a test tile, press **A** (approve) or **R** (reject), then see the truth, the model's class map and its errors. The app scores **you and the model** against the true label. Can you beat the basic model? (Tick *Timed* for an 8-second limit.)

**Explorer mode:** drag the handle over a tile to compare the photo with the model's class map; filter by model mistakes or borderline tiles.

The app runs on Colab callbacks and serves everything from memory — outside Colab you get the inspector instead.

In [ ]:
# @title 🎮 Play: you vs. the model { display-mode: "form" }
launch_warehouse_app(res_basic)

---
## 8 🔭 Crack or stain? — The receptive field

> 🧭 **Why this step?** Section 6 showed the basic model confusing belt and tile, and cracks and stains. A plausible cause: each unit sees too little of the image. We test that **hypothesis** with a controlled experiment — change only the depth, keep everything else fixed — and look at which metrics actually move.

Up close, a crack, a dark stain edge, a marble vein and a scratch all look like the same thing: a thin dark line. What tells them apart is **context**: a crack runs through the tile and breaks its surface, a stain has a soft blotchy outline, a vein belongs to a pattern that continues across the tile. To decide whether a pixel belongs to the damaged zone, a unit must see **the whole mark and the surface around it**.

The **receptive field** of a unit is the region of the input that can influence it. Each layer grows it:

$$r_{\text{out}} = r_{\text{in}} + (k - 1)\cdot j_{\text{in}}, \qquad j_{\text{out}} = j_{\text{in}} \cdot s$$

where $k$ is the kernel size, $s$ the stride and $j$ the distance (in input pixels) between neighbouring units. Plain 3×3 convolutions add only 2 pixels each — **downsampling** (stride or pooling) is what makes the field grow fast, because every later layer then takes bigger steps.

Below we measure the receptive field of the bottleneck (by back-propagating from one bottleneck unit and seeing which input pixels receive a gradient) and compare it with the **size of the damaged zones** in the data.

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet" not in globals():
    unet = load_model("simple_unet")

In [ ]:
# @title Measure the receptive fields { display-mode: "form" }
sizes = defect_sizes(cache, every=5)
rf = {"SimpleUNet": theoretical_receptive_field(unet)}
for d in (2, 3, 4):
    rf[f"DepthUNet depth={d}"] = theoretical_receptive_field(DepthUNet(depth=d))

print("Damaged-zone side length (px): median {:.0f}, 90th percentile {:.0f}, max {:.0f}".format(
    np.median(sizes), np.percentile(sizes, 90), sizes.max()))
for name, r in rf.items():
    print(f"  {name:<18} bottleneck receptive field {r:>4} px   (damaged zones that fit inside: {100 * np.mean(sizes <= r):.0f} %)")

W.rf_visualizer(cache, IDX, rf, sizes)

### 8.1 Experiment: does depth fix it?

`DepthUNet(depth=d)` is a small U-Net whose only knob is the **number of downsampling steps** (every step halves the resolution and doubles the channels; skip connections keep the details). We train depth 2, 3 and 4 with the same recipe.

> **🤔 Predict first:** which depth will have the best IoU and threshold accuracy, and why? Write your guess down before running the cell (~2 min per model on an L4).

In [ ]:
depth_results = {}
for d in (2, 3, 4):
    torch.manual_seed(0)
    model_d = DepthUNet(depth=d, base_ch=16)
    title = f"DepthUNet depth={d} · receptive field {rf[f'DepthUNet depth={d}']} px · {count_parameters(model_d):,} params"
    train_segmentation(model_d, ds_train, ds_val, epochs=EPOCHS(8), lr=2e-3,
                       dashboard=W.TrainingDashboard(title, EPOCHS(8)))
    depth_results[d] = evaluate_models(model_d, ds_test=ds_test, label=f"depth {d}")
    depth_results[d].model = model_d
    save_model(model_d, f"depth_unet_{d}")

pd.DataFrame([{
    "depth": d, "receptive field (px)": rf[f"DepthUNet depth={d}"], "parameters": count_parameters(r.model),
    "mean IoU": round(r.iou, 3), "damaged-zone IoU": round(r.iou_classes["damaged"], 3),
    "usable-share error (pp)": round(r.frac_mae, 2), "threshold accuracy (%)": round(r.acc["Threshold"], 1),
} for d, r in depth_results.items()]).set_index("depth")

In [ ]:
# @title ▶ RESUME HERE — reloads the three depth models (§8) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE for the rest of Section 8 — reloads the three depth models instead of retraining them
if "depth_results" not in globals():
    rf = {f"DepthUNet depth={d}": theoretical_receptive_field(DepthUNet(depth=d)) for d in (2, 3, 4)}
    depth_results = {}
    for d in (2, 3, 4):
        model_d = load_model(f"depth_unet_{d}")
        depth_results[d] = evaluate_models(model_d, ds_test=ds_test, label=f"depth {d}")
        depth_results[d].model = model_d

**Read the table carefully:** which metrics change a lot with depth, and which barely move? Keep your answer for the questions below.

The same test tiles for the shallowest and the deepest model — the ones where depth 2 misses the most damage:

In [ ]:
# @title Compare the models on the same tiles { display-mode: "form" }
W.compare_gallery([depth_results[2], depth_results[4]], n=6, title="Depth 2 vs depth 4 on depth 2's hardest tiles",
                  names=["DepthUNet, depth 2", "DepthUNet, depth 4"])

### 8.2 Theoretical vs. effective receptive field

The *theoretical* field says which pixels **can** influence a unit. The *effective* field shows how much they actually do in the trained network: we average the input-gradient magnitude over a few real test images. The square marks the smallest centred region holding 90 % of the gradient.

In [ ]:
# @title Show the effective receptive fields { display-mode: "form" }
imgs = [ds_test[i][0] for i in range(8)]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, d in zip(axes, (2, 3, 4)):
    side, grad = effective_receptive_field(depth_results[d].model, imgs, mass=0.9)
    theo = rf[f"DepthUNet depth={d}"]
    c, half = grad.shape[0] // 2, 56                       # show a 113×113 px window around the unit
    ax.imshow((grad ** 0.5)[c - half:c + half + 1, c - half:c + half + 1], cmap="Blues",
              extent=(-half - 0.5, half + 0.5, half + 0.5, -half - 0.5))
    ax.add_patch(plt.Rectangle((-side / 2, -side / 2), side, side, fill=False, color="#eb6834", lw=2))
    ax.add_patch(plt.Rectangle((-theo / 2, -theo / 2), theo, theo, fill=False, color="#52514e", lw=1))
    ax.set_title(f"depth {d}: effective ≈ {side} px (orange) · theoretical {theo} px (grey)", fontsize=10)
    ax.axis("off")
plt.suptitle("Where does one bottleneck unit look? (√ of the input-gradient magnitude)")
plt.tight_layout(); plt.show()

#### Questions
1. Was your prediction right? Which metrics improve with depth, and why does the accept/reject accuracy change so much less than the IoU?
2. Why is the effective receptive field smaller than the theoretical one?
3. Two other ways to enlarge the receptive field *without* more downsampling are **dilated convolutions** and **larger kernels**. What would each cost?
4. Parameters also grew with depth. Design an experiment that separates "bigger receptive field" from "more parameters".

*Your answers:*

---
## 9 🏆 Challenge: beat the targets!

> 🧭 **Why this step?** With the diagnosis in hand you combine the fixes — skip connections, depth, normalisation, a better loss, augmentation — and add a regressor that predicts the number the business rule is about. Then we look at what errors **cost**, not just how many there are.

Redesign the networks and hit these test-set targets:

| Method | Target |
|---|---|
| Threshold (85 % rule on your class map) | **≥ 96 %** |
| FNN head | **≥ 78 %** |
| CNN head | **≥ 92 %** |
| **Usable-share regressor** (new, 9.4) | **≥ 90 %** |
| Mean IoU (3 classes) / usable-share error | **≥ 0.90** / **≤ 2.5 points** |

*The targets come from a full run of the reference solution; the basic models from Sections 3–5 reach 86 % / 65 % / 71 %.*

**You can change anything:** depth and width, block design, skip connections, normalisation, loss (cross-entropy, Dice, both, class weights), optimiser, learning rate, epochs, data augmentation (`ds_train_aug`).

**Provided for you:** `train_segmentation(model, ds_train, ds_val, …)` and `train_head(backbone, head, ds_train, ds_val, mode=…)` (with live dashboards, validation metrics, a cosine learning-rate schedule and mixed precision on GPU), plus all evaluation panels.

**Rules**
1. Define `ImprovedUNet`, `ImprovedFNN`, `ImprovedCNN` and `UsableShareRegressor` yourself in this notebook (`DepthUNet` in `utils.py` is for the experiment only — don't just import it).
2. The U-Net inherits from `LatentUNetBase` and implements `encode` / `decode`; heads take the bottleneck (`(B, C)` for the FNN, `(B, C, H', W')` for the others) and return shape `(B,)`.
3. The heads depend on the U-Net's bottleneck: **retrain all heads** after changing the U-Net.

### 9.1 Improved U-Net

The `SimpleUNet` misses the key ingredient that gives U-Net its name. Other hints:
- **Skip connections**: concatenate encoder features with the upsampled decoder features at the same resolution — the decoder gets the fine details back. Watch the input channels of the decoder blocks!
- **Depth**: what did Section 8 tell you about the receptive field you need?
- **Blocks**: two convolutions per block, **BatchNorm** after each convolution.

In [ ]:
class ConvBlock(nn.Module):
    """Convolution block — 🎯 TODO: improve it (BatchNorm? two convolutions?)"""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ImprovedUNet(LatentUNetBase):
    """🎯 TODO: your improved U-Net."""

    def __init__(self, in_ch=1, base_ch=32):
        super().__init__()
        # TODO: define encoder blocks, downsampling, bottleneck, upsampling, decoder blocks and out_conv.
        # Tip: nn.ModuleList lets you build a variable number of levels in a loop.
        self.latent_dim = ...   # number of channels in YOUR bottleneck
        raise NotImplementedError("Build your improved U-Net here!")

    def encode(self, x):
        # TODO: return (bottleneck, skips) — skips = encoder outputs, one per resolution level
        raise NotImplementedError

    def decode(self, b, skips):
        # TODO: upsample, concatenate the matching skip (torch.cat([...], dim=1)), convolve — repeat — out_conv
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
unet_improved = ImprovedUNet(in_ch=1, base_ch=32).to(device)
W.architecture_diagram(unet_improved, title="ImprovedUNet")
print(f"Bottleneck receptive field: {theoretical_receptive_field(unet_improved)} px")

Train it. 🎯 This is where you tune: epochs, learning rate, Dice, augmentation. The dashboard shows validation IoU and threshold accuracy after every epoch.

In [ ]:
unet_history = train_segmentation(
    unet_improved, ds_train, ds_val,     # 🎯 try ds_train_aug (random flips / rotations)
    epochs=EPOCHS(20), lr=1e-3,           # 🎯 more epochs? another learning rate?
    use_dice=False,                       # 🎯 would cross-entropy + Dice help with the small damaged zones?
    dashboard=W.TrainingDashboard("Improved U-Net", EPOCHS(20), target_acc=96),
)

In [ ]:
# @title 💾 Save the model { display-mode: "form" }
save_model(unet_improved, "improved_unet")

### 9.2 Improved FNN head

`SimpleApprovalFNN` was logistic regression on the pooled vector. Input: `(B, latent_dim)`, output: `(B,)` probabilities (end with a Sigmoid).

In [ ]:
# @title ▶ RESUME HERE — reloads the improved U-Net (§9.1) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet_improved" not in globals():
    unet_improved = load_model("improved_unet")

In [ ]:
class ImprovedFNN(nn.Module):
    def __init__(self, latent_dim=512):
        super().__init__()
        # TODO 🎯: hidden layer(s)? non-linearity? dropout?
        raise NotImplementedError("Define ImprovedFNN")

    def forward(self, z):
        """z: (B, latent_dim) → (B,) probabilities"""
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
classifier_fnn_improved = ImprovedFNN(latent_dim=unet_improved.latent_dim)
fnn_history = train_head(unet_improved, classifier_fnn_improved, ds_train, ds_val, mode="fnn",
                         epochs=EPOCHS(20), lr=1e-3,
                         dashboard=W.TrainingDashboard("Improved FNN head", EPOCHS(20), target_acc=78))
save_model(classifier_fnn_improved, "improved_fnn")

> 💡 **Why does the FNN plateau around 80 %?** Global average pooling turns the bottleneck into an *average feature*: it keeps "how much crack-like texture is there", but loses **how big the tile is and how much of it the damage covers**. The 85 % rule is a *ratio of two areas*, which a single averaged vector can only approximate. The CNN and the regressor keep the spatial map and do much better — the same lesson as Sections 4–5, now at full scale.

### 9.3 Improved CNN head

Input: the bottleneck map `(B, latent_dim, H', W')`, output: `(B,)` probabilities.

In [ ]:
# @title ▶ RESUME HERE — reloads the improved U-Net (§9.1) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet_improved" not in globals():
    unet_improved = load_model("improved_unet")

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()
        # TODO 🎯: more conv layers? BatchNorm? pooling? more filters?
        raise NotImplementedError("Define ImprovedCNN")

    def forward(self, z_map):
        """z_map: (B, in_channels, H', W') → (B,) probabilities"""
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
classifier_cnn_improved = ImprovedCNN(in_channels=unet_improved.latent_dim)
cnn_history = train_head(unet_improved, classifier_cnn_improved, ds_train, ds_val, mode="cnn",
                         epochs=EPOCHS(20), lr=1e-3,
                         dashboard=W.TrainingDashboard("Improved CNN head", EPOCHS(20), target_acc=92))
save_model(classifier_cnn_improved, "improved_cnn")

### 9.4 New: a usable-share regressor

The FNN and CNN learn a **yes/no answer**. But the business rule is about a **number** — the usable share of the tile — and the 85 % cut is a fixed rule. Instead of learning the decision, we can learn the number and keep the rule explicit:

```
bottleneck map ──► UsableShareRegressor ──► predicted usable share ∈ [0, 1] ──► ≥ 85 % ?  ──► APPROVE / REJECT
```

It is trained with an **L1 loss** against the true usable share of each tile (`train_head(..., mode="reg")`), so every training example says *how far* it is from the line, not only which side it is on. The prediction is also easy to explain to a warehouse manager ("91 % of this tile is usable").

**Design hint.** The usable share is a ratio of two areas: usable surface ÷ tile surface. A neat architecture predicts, **for every bottleneck location**, how much tile it covers and how much of that is usable (1×1 convolutions → two sigmoids), then divides the sums — the output is automatically between 0 and 1. A simpler version (one sigmoid, averaged) also works.

In [ ]:
# @title ▶ RESUME HERE — reloads the improved U-Net (§9.1) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "unet_improved" not in globals():
    unet_improved = load_model("improved_unet")

In [ ]:
class UsableShareRegressor(nn.Module):
    def __init__(self, in_channels=512):
        super().__init__()
        # TODO 🎯: layers that map (B, in_channels, H', W') to one number per image in [0, 1]
        raise NotImplementedError("Define UsableShareRegressor")

    def forward(self, z_map):
        """z_map: (B, in_channels, H', W') → (B,) predicted usable share in [0, 1]"""
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
regressor = UsableShareRegressor(in_channels=unet_improved.latent_dim)
reg_history = train_head(unet_improved, regressor, ds_train, ds_val, mode="reg", epochs=EPOCHS(20), lr=1e-3,
                         dashboard=W.TrainingDashboard("Usable-share regressor", EPOCHS(20), target_acc=90))
save_model(regressor, "improved_regressor")

### 9.5 Final evaluation

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3), the FNN head (§4), the CNN head (§5), the improved U-Net (§9.1), the improved heads (§9.2–9.4) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "regressor" not in globals():
    unet_improved, classifier_fnn_improved, classifier_cnn_improved, regressor = load_models(
        "improved_unet", "improved_fnn", "improved_cnn", "improved_regressor")
if "res_basic" not in globals():            # the comparison gallery below also needs the basic models
    if "classifier_cnn" not in globals():
        unet, classifier_fnn, classifier_cnn = load_models("simple_unet", "simple_fnn", "simple_cnn")
    res_basic = evaluate_models(unet, classifier_fnn, classifier_cnn, ds_test, label="Basic models")

In [ ]:
res_improved = evaluate_models(unet_improved, classifier_fnn_improved, classifier_cnn_improved, ds_test,
                               regressor=regressor, label="Improved models")
board.record(res_improved)

In [ ]:
# @title Show where the models go wrong { display-mode: "form" }
W.confusion_explorer(res_improved)

In [ ]:
# @title Open the test-set inspector { display-mode: "form" }
W.test_inspector(res_improved, max_items=60)

In [ ]:
# @title Compare the models on the same tiles { display-mode: "form" }
W.compare_gallery([res_basic, res_improved], n=6, title="Basic vs improved U-Net on the basic U-Net's hardest tiles",
                  names=["Basic U-Net (§3)", "Improved U-Net (§9.1)"])

### 9.6 From accuracy to business cost

So far every mistake counted the same. In the warehouse they don't: a **false approval** sells a damaged tile to a customer (returns, complaints, reputation), a **false rejection** sends a sellable tile to the recycling bin (lost revenue). Drag the decision line and change the two costs.

In [ ]:
# @title ▶ RESUME HERE — reloads the improved U-Net (§9.1), the improved heads (§9.2–9.4) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "regressor" not in globals():
    unet_improved, classifier_fnn_improved, classifier_cnn_improved, regressor = load_models(
        "improved_unet", "improved_fnn", "improved_cnn", "improved_regressor")
if "res_improved" not in globals():
    res_improved = evaluate_models(unet_improved, classifier_fnn_improved, classifier_cnn_improved, ds_test,
                                   regressor=regressor, label="Improved models")

In [ ]:
# @title Open the threshold and business-cost explorer { display-mode: "form" }
W.threshold_explorer(res_improved, cost_false_approve=5, cost_false_reject=1)

#### Questions
1. With a false approval 5× as expensive as a false rejection, where is the cheapest cut? Why is it not at 85 %?
2. Switch to *Regression usable share*. Are its errors distributed differently from the usable share of the U-Net's class map?
3. Which of your four methods would you deploy, and what would you tell the warehouse manager about its failure cases?

*Your answers:*

### 9.7 The app with your improved model

In [ ]:
# @title 🎮 Play: you vs. the model { display-mode: "form" }
launch_warehouse_app(res_improved)

---
## 10 🎁 Bonus: transfer learning with a pretrained encoder

> 🧭 **Why this step?** Your encoder learned to see from 3 500 photos. Would a network that has already seen 1.2 million photos do better with the same data? This is the standard move in industry when labelled data is limited.

Your encoder learned to see from 3 500 synthetic photos. A **ResNet-18** trained on 1.2 million ImageNet photos already knows edges, textures and shapes. Here we use it as the U-Net **encoder** and only design the decoder:

| ResNet stage | Resolution | Channels | Used as |
|---|---|---|---|
| stem (conv 7×7, stride 2) | 128×128 | 64 | skip 0 |
| layer1 (after max-pool) | 64×64 | 64 | skip 1 |
| layer2 | 32×32 | 128 | skip 2 |
| layer3 | 16×16 | 256 | skip 3 |
| layer4 | **8×8** | **512** | **bottleneck** (receptive field > the whole image) |

ImageNet models expect 3 normalised colour channels, so the grey image is repeated 3× and normalised with the ImageNet mean/std inside `encode`.

**Two-stage training:** first train only the new decoder with the encoder **frozen** (the pretrained features are not disturbed by the random gradients of a fresh decoder), then **unfreeze** everything and fine-tune with a smaller learning rate.

In [ ]:
# @title ▶ RESUME HERE — reloads your ConvBlock (§9.1), the improved U-Net (§9.1) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "ConvBlock" not in globals():             # your block from 9.1 (UpBlock below uses it)
    ConvBlock = class_from_checkpoint("improved_unet", "ConvBlock")

In [ ]:
import torchvision


class UpBlock(nn.Module):
    """Upsample ×2, concatenate a skip (optional), then a ConvBlock."""

    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)

    def forward(self, x, skip=None):
        # TODO: upsample x; if a skip is given, concatenate it along the channel dimension; apply self.conv
        raise NotImplementedError  # YOUR CODE HERE


class ResNetUNet(LatentUNetBase):
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        r = torchvision.models.resnet18(weights=weights)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.register_buffer("mean", torch.tensor(self.MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(self.STD).view(1, 3, 1, 1))
        self.latent_dim = 512

        self.up4 = UpBlock(512, 256, 256)   #  8 → 16, skip layer3
        self.up3 = UpBlock(256, 128, 128)   # 16 → 32, skip layer2
        self.up2 = UpBlock(128, 64, 64)     # 32 → 64, skip layer1
        self.up1 = UpBlock(64, 64, 32)      # 64 → 128, skip stem
        self.up0 = UpBlock(32, 0, 16)       # 128 → 256, no skip
        self.out_conv = nn.Conv2d(16, 3, kernel_size=1)     # belt / usable / damaged

    def set_encoder_trainable(self, flag):
        for m in (self.stem, self.layer1, self.layer2, self.layer3, self.layer4):
            for p in m.parameters():
                p.requires_grad = flag

    def encode(self, x):
        x = (x.repeat(1, 3, 1, 1) - self.mean) / self.std     # grey → ImageNet-normalised RGB
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        return self.layer4(s3), [s0, s1, s2, s3]

    def decode(self, b, skips):
        # TODO: s0, s1, s2, s3 = skips; go up4(b, s3) → up3(·, s2) → up2(·, s1) → up1(·, s0) → up0(·) → out_conv
        raise NotImplementedError  # YOUR CODE HERE

In [ ]:
torch.manual_seed(0)
resnet_unet = ResNetUNet(pretrained=not FAST_DEV_RUN).to(device)
W.architecture_diagram(resnet_unet, title="ResNet-18 U-Net")

# Stage A — encoder frozen: only the decoder learns
resnet_unet.set_encoder_trainable(False)
train_segmentation(resnet_unet, ds_train_aug, ds_val, epochs=EPOCHS(5), lr=1e-3,
                   dashboard=W.TrainingDashboard("ResNet U-Net · stage A: frozen encoder", EPOCHS(5), target_acc=96))

# Stage B — everything trainable, smaller learning rate
resnet_unet.set_encoder_trainable(True)
train_segmentation(resnet_unet, ds_train_aug, ds_val, epochs=EPOCHS(15), lr=3e-4,
                   dashboard=W.TrainingDashboard("ResNet U-Net · stage B: fine-tuning", EPOCHS(15), target_acc=96))
save_model(resnet_unet, "resnet_unet")

In [ ]:
# @title ▶ RESUME HERE — reloads the improved heads (§9.2–9.4), the ResNet U-Net (§10) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "resnet_unet" not in globals():
    resnet_unet = load_model("resnet_unet")
if "UsableShareRegressor" not in globals():     # your class from 9.4, recovered from its checkpoint
    UsableShareRegressor = class_from_checkpoint("improved_regressor")

In [ ]:
torch.manual_seed(0)
regressor_resnet = UsableShareRegressor(in_channels=resnet_unet.latent_dim)
train_head(resnet_unet, regressor_resnet, ds_train, ds_val, mode="reg", epochs=EPOCHS(15), lr=1e-3,
           dashboard=W.TrainingDashboard("Usable-share regressor on the ResNet bottleneck", EPOCHS(15), target_acc=90))
save_model(regressor_resnet, "resnet_regressor")

res_resnet = evaluate_models(resnet_unet, ds_test=ds_test, regressor=regressor_resnet, label="ResNet-18 U-Net")
board.record(res_resnet)

In [ ]:
# @title ▶ RESUME HERE — reloads the basic U-Net (§3), the FNN head (§4), the CNN head (§5), the improved U-Net (§9.1), the improved heads (§9.2–9.4), the ResNet U-Net (§10) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "res_resnet" not in globals():
    resnet_unet, regressor_resnet = load_models("resnet_unet", "resnet_regressor")
    res_resnet = evaluate_models(resnet_unet, ds_test=ds_test, regressor=regressor_resnet, label="ResNet-18 U-Net")
if "classifier_cnn" not in globals():
    unet, classifier_fnn, classifier_cnn = load_models("simple_unet", "simple_fnn", "simple_cnn")
if "res_basic" not in globals():
    res_basic = evaluate_models(unet, classifier_fnn, classifier_cnn, ds_test, label="Basic models")
if "regressor" not in globals():
    unet_improved, classifier_fnn_improved, classifier_cnn_improved, regressor = load_models(
        "improved_unet", "improved_fnn", "improved_cnn", "improved_regressor")
if "res_improved" not in globals():
    res_improved = evaluate_models(unet_improved, classifier_fnn_improved, classifier_cnn_improved, ds_test,
                                   regressor=regressor, label="Improved models")

In [ ]:
# @title Compare the models on the same tiles { display-mode: "form" }
W.compare_gallery([res_basic, res_improved, res_resnet], n=6, title="Basic vs improved vs pretrained-encoder U-Net",
                  names=["Basic U-Net (§3)", "Improved U-Net (§9.1)", "ResNet-18 U-Net (§10)"])

#### Questions
1. Compare stage A and stage B on the dashboards. What did fine-tuning add?
2. The ResNet was trained on colour photos of dogs and cars. Why does it still help on grey photos of tiles?
3. When would you **not** use a pretrained encoder?

*Your answers:*

---
## 11 🏭 Deploy: your model in the warehouse inspection platform

> 🧭 **Why this step?** A model only creates value when someone uses it. Here your best model leaves the notebook and runs in the warehouse's inspection platform — the last step from experiment to product.

The warehouse doesn't run notebooks — it runs an **inspection platform**: a web app where operators upload tile photos (or the camera feeds them in), every tile gets a class map, a usable share and a verdict, borderline tiles land in a **review queue** for a human, and managers follow approval rates and recovered value on a dashboard.

The platform lives in `week3/project1/platform/` and runs **on your own laptop** (it does not run inside Colab). It comes with a model trained by the course staff — here you swap in **your own**:

1. pick your best segmentation model (check the scoreboard above),
2. run the next cells: they export the model as **TorchScript** (a compiled network that loads on any computer with PyTorch, without this notebook's code) and download `platform_model.pt`,
3. start the platform locally (see `week3/project1/platform/README.md`) and upload the file under **Settings → Model**.

In [ ]:
# @title ▶ RESUME HERE — reloads the improved U-Net (§9.1), the ResNet U-Net (§10) (only needed after a restart) { display-mode: "form" }
# ▶ RESUME HERE — only needed after a restart (first run the 🔁 cells of Sections 1–2)
if "res_resnet" not in globals() and has_checkpoint("resnet_unet"):
    resnet_unet, regressor_resnet = load_models("resnet_unet", "resnet_regressor")
    res_resnet = evaluate_models(resnet_unet, ds_test=ds_test, regressor=regressor_resnet, label="ResNet-18 U-Net")
if "res_improved" not in globals():
    unet_improved = load_model("improved_unet")
    res_improved = evaluate_models(unet_improved, ds_test=ds_test, label="Improved models")

In [ ]:
# @title 🏭 Export and download your best model for the platform { display-mode: "form" }
# 🎯 your best segmentation model and its test result — check the scoreboard
best_model, best_res = (resnet_unet, res_resnet) if "res_resnet" in globals() else (unet_improved, res_improved)

path = export_for_platform(best_model, f"{WORK_DIR}/platform_model.pt", name="Our team's model", result=best_res)
if IN_COLAB:
    from google.colab import files
    files.download(path)              # your browser saves platform_model.pt
else:
    print("Your model file:", os.path.abspath(path))

---
## 12 📌 Takeaways

Summarise in your own words (2–3 sentences each):

Your **scoreboard** (Section 6 onwards) records the whole journey — look at how each step moved the numbers.

1. What did the **receptive field** experiment teach you about architecture design?
2. **Threshold vs FNN vs CNN vs regressor** — which would you deploy and why?
3. What would you check before putting this system into a real warehouse?

*Your answers:*